# AnyUp3D — Experiment 2: Temporal Consistency
**Metric:** Adjacent frame cosine similarity | 2D AnyUp vs AnyUp3D  
**Dataset:** DAVIS 2017 validation set

## Cell 0 — Installs & Configuration

In [1]:
# ── CELL 0 ───────────────────────────────────────────────────────────────────
# Installs & configuration
from google.colab import drive
drive.mount('/content/drive')
# !pip install natten --quiet
# !pip install transformers --quiet

import os, sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from pathlib import Path
from PIL import Image
from sklearn.decomposition import PCA

# ── repo ──────────────────────────────────────────────────────────────────────
REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR} — skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# ── paths ─────────────────────────────────────────────────────────────────────
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"
OUTPUT_DIR   = "/content/drive/MyDrive/results/exp2"
CACHE_DIR    = os.path.join(OUTPUT_DIR, "swin_cache")
DAVIS_ROOT   = "/content/DAVIS"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)

# ── sampling ──────────────────────────────────────────────────────────────────
N_FRAMES = 16    # exact frames fed to VideoMAE — do not change without rechecking
                 # VideoMAE requires exactly 16; clips shorter than this are skipped

# ── model / feature config ────────────────────────────────────────────────────
IMG_SIZE  = 224  # spatial input resolution
FEAT_DIM  = 768  # VideoMAE-base output channels
FEAT_SIZE = 14   # = IMG_SIZE / 16 with VideoMAE patch_size=16; depends on IMG_SIZE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── TStage stub ───────────────────────────────────────────────────────────────
# Must be defined before torch.load — checkpoint embeds this object via pickle
class TStage:
    def __init__(self, t=None, start_step=None, batch_size=None):
        self.t          = t
        self.start_step = start_step
        self.batch_size = batch_size

Mounted at /content/drive
Cloned to /content/anyup
Working directory: /content/anyup
Device: cuda


## Cell 1 — Model Loading: Video Swin-T | AnyUp3D | 2D AnyUp

In [2]:
# ── CELL 1 ───────────────────────────────────────────────────────────────────
# Model loading: VideoMAE | AnyUp3D | 2D AnyUp

sys.path.insert(0, "/content/anyup")
from anyup.model    import AnyUp
from anyup.model_2d import AnyUp2D  # import directly — do NOT use torch.hub.load for 2D

from transformers import VideoMAEModel

# ── VideoMAE ──────────────────────────────────────────────────────────────────
videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(device).eval()
print("VideoMAE loaded")

# ── AnyUp3D ───────────────────────────────────────────────────────────────────
model_3d = AnyUp(
    input_dim       = 3,
    qk_dim          = 128,
    kernel_size     = 1,
    kernel_size_lfu = 5,
    window_ratio    = 0.1,
    num_heads       = 4,
    t_k             = 3,
    window_t        = 4,
).to(device).eval()

ckpt             = torch.load(CKPT_ANYUP3D, map_location="cpu", weights_only=False)  # weights_only=False required for TStage
sd               = ckpt["anyup"]   # key is "anyup" — not "model" or "state_dict"
missing, unexpected = model_3d.load_state_dict(sd, strict=True)
print(f"AnyUp3D loaded | missing={missing} | unexpected={unexpected}")

# ── 2D AnyUp ─────────────────────────────────────────────────────────────────
model_2d = AnyUp2D().to(device).eval()
sd_2d    = torch.hub.load_state_dict_from_url(
    "https://github.com/wimmerth/anyup/releases/download/checkpoint/anyup_paper.pth",
    map_location=device,
)
model_2d.load_state_dict(sd_2d)
print("AnyUp2D loaded")

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias          

VideoMAE loaded
AnyUp3D loaded | missing=[] | unexpected=[]
Downloading: "https://github.com/wimmerth/anyup/releases/download/checkpoint/anyup_paper.pth" to /root/.cache/torch/hub/checkpoints/anyup_paper.pth


100%|██████████| 3.38M/3.38M [00:00<00:00, 136MB/s]

AnyUp2D loaded


## Cell 2 — DAVIS 2017 Download & Clip Loader

> **Manual fallback:** If `wget` fails below:
> 1. Download `DAVIS-2017-trainval-480p.zip` from [davischallenge.org](https://davischallenge.org)
> 2. Upload to Drive or directly to `/content/`
> 3. Set `DAVIS_ROOT` in Cell 0 to the extracted folder path
> 4. Skip this cell and go to Cell 3

In [3]:
# ── CELL 2 ───────────────────────────────────────────────────────────────────
# DAVIS 2017 evaluation package + dataset extraction

# ── evaluation package (also needed for Exp 3 J&F scoring) ───────────────────
if not os.path.exists("/content/davis2017-evaluation"):
    subprocess.run([
        "git", "clone",
        "https://github.com/davisvideochallenge/davis2017-evaluation.git",
        "/content/davis2017-evaluation"
    ], check=True)
    subprocess.run(
        ["python", "setup.py", "install"],
        cwd="/content/davis2017-evaluation", check=True
    )
    print("DAVIS evaluation package installed")
else:
    print("DAVIS evaluation package already present")

# ── dataset extraction ────────────────────────────────────────────────────────
DAVIS_ZIP = "/content/drive/MyDrive/results/DAVIS-2017-trainval-480p.zip"

if not os.path.exists(os.path.join(DAVIS_ROOT, "ImageSets")):
    print("Extracting DAVIS from Drive...")
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)
    print("Done.")
else:
    print(f"DAVIS already present at {DAVIS_ROOT}")

# ── clip loader ───────────────────────────────────────────────────────────────
def load_clip_paths(davis_root, split="val"):
    val_txt    = Path(davis_root) / "ImageSets" / "2017" / f"{split}.txt"
    clip_names = [l.strip() for l in val_txt.read_text().splitlines() if l.strip()]

    clips = {}
    for name in clip_names:
        frame_dir  = Path(davis_root) / "JPEGImages" / "480p" / name
        all_frames = sorted(frame_dir.glob("*.jpg"))
        if len(all_frames) < N_FRAMES:   # need at least 16 frames; set by N_FRAMES in Cell 0
            print(f"  skip {name} — only {len(all_frames)} frames available")
            continue
        # evenly spaced indices → exactly N_FRAMES frames covering the full clip
        indices        = np.linspace(0, len(all_frames) - 1, N_FRAMES, dtype=int)
        clips[name]    = [str(all_frames[i]) for i in indices]

    print(f"\nLoaded {len(clips)} clips (each T={N_FRAMES})")
    for name in clips:
        print(f"  {name}")
    return clips

if os.path.exists(os.path.join(DAVIS_ROOT, "ImageSets")):
    clips = load_clip_paths(DAVIS_ROOT)
else:
    print("DAVIS not found — check DAVIS_ZIP path and re-run this cell.")
    clips = {}

clips = dict(list(clips.items()))  # ← limit to 3 clips for debugging; remove for full run
print(f"\nRunning on: {list(clips.keys())}")

DAVIS evaluation package installed
Extracting DAVIS from Drive...
Done.

Loaded 30 clips (each T=16)
  bike-packing
  blackswan
  bmx-trees
  breakdance
  camel
  car-roundabout
  car-shadow
  cows
  dance-twirl
  dog
  dogs-jump
  drift-chicane
  drift-straight
  goat
  gold-fish
  horsejump-high
  india
  judo
  kite-surf
  lab-coat
  libby
  loading
  mbike-trick
  motocross-jump
  paragliding-launch
  parkour
  pigs
  scooter-black
  shooting
  soapbox

Running on: ['bike-packing', 'blackswan', 'bmx-trees', 'breakdance', 'camel', 'car-roundabout', 'car-shadow', 'cows', 'dance-twirl', 'dog', 'dogs-jump', 'drift-chicane', 'drift-straight', 'goat', 'gold-fish', 'horsejump-high', 'india', 'judo', 'kite-surf', 'lab-coat', 'libby', 'loading', 'mbike-trick', 'motocross-jump', 'paragliding-launch', 'parkour', 'pigs', 'scooter-black', 'shooting', 'soapbox']


In [4]:
import shutil; shutil.rmtree(CACHE_DIR); os.makedirs(CACHE_DIR, exist_ok=True)


## Cell 3 — Swin Feature Precomputation → Cache per Clip
Run once; subsequent runs skip already-cached clips.

In [5]:
# ── CELL 3 ───────────────────────────────────────────────────────────────────
# VideoMAE feature precomputation → cache per clip to Drive
# Run once; subsequent runs skip already-cached clips
#
# Clear old 7×7 Swin cache before running for the first time:
# import shutil; shutil.rmtree(CACHE_DIR); os.makedirs(CACHE_DIR, exist_ok=True)

preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),   # depends on IMG_SIZE in Cell 0
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_videomae_features(frame_paths):
    """
    frame_paths : exactly N_FRAMES=16 paths (evenly sampled from clip)
    Returns:
        feats  : (8, FEAT_DIM, FEAT_SIZE, FEAT_SIZE)  — 8 temporal tokens (tubelet_size=2)
        guides : (8, 3, IMG_SIZE, IMG_SIZE)            — every 2nd frame to match feat T
    """
    frames = [preprocess(Image.open(p).convert("RGB")) for p in frame_paths]
    frames = torch.stack(frames, dim=0)                     # (16, 3, 224, 224)

    # VideoMAE expects (B, T, C, H, W)
    inp = frames.unsqueeze(0).to(device)                    # (1, 16, 3, 224, 224); T=16 required

    with torch.no_grad():
        out = videomae(pixel_values=inp)

    # last_hidden_state: (1, 8*14*14, 768) = (1, 1568, 768)
    tokens = out.last_hidden_state.squeeze(0).cpu()         # (1568, 768)
    feats  = tokens.reshape(8, 14, 14, 768)\
                   .permute(0, 3, 1, 2).contiguous()        # (8, 768, 14, 14)

    # subsample frames by tubelet_size=2 to align guide T with feat T
    guides = frames[::2]                                    # (8, 3, 224, 224)

    return feats, guides

for clip_name, frame_paths in clips.items():
    cache_path = os.path.join(CACHE_DIR, f"{clip_name}.pt")
    if os.path.exists(cache_path):
        print(f"  skip {clip_name} (cached)")
        continue
    feats, guides = extract_videomae_features(frame_paths)
    torch.save({"feats": feats, "guides": guides}, cache_path)
    print(f"  {clip_name}: feats={tuple(feats.shape)}, guides={tuple(guides.shape)}")
    del feats, guides
    torch.cuda.empty_cache()

print("\nPrecomputation complete.")

  bike-packing: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  blackswan: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  bmx-trees: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  breakdance: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  camel: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  car-roundabout: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  car-shadow: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  cows: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  dance-twirl: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  dog: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  dogs-jump: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  drift-chicane: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  drift-straight: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  goat: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  gold-fish: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  horsejump-high: feats=(8, 768, 14, 14), guides=(8, 3, 224, 224)
  india: feats=(8,

## Cell 4 — AnyUp3D Full-Volume Inference (all T frames at once)

In [6]:
# import glob

# files = glob.glob(os.path.join(CACHE_DIR, "*_3d.pt"))
# for f in files:
#     os.remove(f)
#     print(f"Deleted: {f}")

# print("Done — re-run Cell 4 now.")

In [7]:
# ── CELL 4 ───────────────────────────────────────────────────────────────────
# AnyUp3D full-volume inference (all T=8 frames at once)
# Saves output immediately per clip to avoid OOM — safe to re-run after crash

def run_anyup3d(feats, guides):
    """
    feats  : (8, 768, 14, 14)
    guides : (8, 3, 224, 224)
    Returns: (8, 768, 224, 224)
    """
    # rearrange to (1, C, T, H, W) for the model
    feat_vol  = feats.permute(1, 0, 2, 3).unsqueeze(0).to(device)    # (1, 768, 8, 14, 14)
    guide_vol = guides.permute(1, 0, 2, 3).unsqueeze(0).to(device)   # (1, 3, 8, 224, 224)
    with torch.no_grad():
        out = model_3d(guide_vol, feat_vol)                           # guide first, features second
    return out.squeeze(0).permute(1, 0, 2, 3).cpu()                  # (8, 768, 224, 224)

for clip_name in clips:
    out_path = os.path.join(CACHE_DIR, f"{clip_name}_3d.pt")
    if os.path.exists(out_path):
        print(f"  skip {clip_name} (cached)")
        continue
    cache_path = os.path.join(CACHE_DIR, f"{clip_name}.pt")
    data = torch.load(cache_path, map_location="cpu")
    out  = run_anyup3d(data["feats"], data["guides"])
    torch.save(out, out_path)
    print(f"  {clip_name}: out={tuple(out.shape)}")
    del data, out
    torch.cuda.empty_cache()

print("\nAnyUp3D inference complete.")

  bike-packing: out=(8, 768, 224, 224)
  blackswan: out=(8, 768, 224, 224)
  bmx-trees: out=(8, 768, 224, 224)
  breakdance: out=(8, 768, 224, 224)
  camel: out=(8, 768, 224, 224)
  car-roundabout: out=(8, 768, 224, 224)
  car-shadow: out=(8, 768, 224, 224)
  cows: out=(8, 768, 224, 224)
  dance-twirl: out=(8, 768, 224, 224)
  dog: out=(8, 768, 224, 224)
  dogs-jump: out=(8, 768, 224, 224)
  drift-chicane: out=(8, 768, 224, 224)
  drift-straight: out=(8, 768, 224, 224)
  goat: out=(8, 768, 224, 224)
  gold-fish: out=(8, 768, 224, 224)
  horsejump-high: out=(8, 768, 224, 224)
  india: out=(8, 768, 224, 224)
  judo: out=(8, 768, 224, 224)
  kite-surf: out=(8, 768, 224, 224)
  lab-coat: out=(8, 768, 224, 224)
  libby: out=(8, 768, 224, 224)
  loading: out=(8, 768, 224, 224)
  mbike-trick: out=(8, 768, 224, 224)
  motocross-jump: out=(8, 768, 224, 224)
  paragliding-launch: out=(8, 768, 224, 224)
  parkour: out=(8, 768, 224, 224)
  pigs: out=(8, 768, 224, 224)
  scooter-black: out=(8, 768,

## Cell 5 — 2D AnyUp Frame-by-Frame Inference (no temporal context)

In [8]:
# ── CELL 5 ───────────────────────────────────────────────────────────────────
# 2D AnyUp frame-by-frame inference (no temporal context)
# Saves output immediately per clip to avoid OOM — safe to re-run after crash

def run_anyup2d(feats, guides):
    """
    feats  : (8, 768, 14, 14)
    guides : (8, 3, 224, 224)
    Returns: (8, 768, 224, 224)
    """
    results = []
    for t in range(feats.shape[0]):        # iterate over T=8 frames independently
        feat_t  = feats[t].unsqueeze(0).to(device)    # (1, 768, 14, 14)
        guide_t = guides[t].unsqueeze(0).to(device)   # (1, 3, 224, 224)
        with torch.no_grad():
            out_t = model_2d(guide_t, feat_t)          # guide first, features second
        results.append(out_t.squeeze(0).cpu())         # (768, 224, 224)
    return torch.stack(results, dim=0)                 # (8, 768, 224, 224)

for clip_name in clips:
    out_path = os.path.join(CACHE_DIR, f"{clip_name}_2d.pt")
    if os.path.exists(out_path):
        print(f"  skip {clip_name} (cached)")
        continue
    cache_path = os.path.join(CACHE_DIR, f"{clip_name}.pt")
    data = torch.load(cache_path, map_location="cpu")
    out  = run_anyup2d(data["feats"], data["guides"])
    torch.save(out, out_path)
    print(f"  {clip_name}: out={tuple(out.shape)}")
    del data, out
    torch.cuda.empty_cache()

print("\n2D AnyUp inference complete.")

  bike-packing: out=(8, 768, 224, 224)
  blackswan: out=(8, 768, 224, 224)
  bmx-trees: out=(8, 768, 224, 224)
  breakdance: out=(8, 768, 224, 224)
  camel: out=(8, 768, 224, 224)
  car-roundabout: out=(8, 768, 224, 224)
  car-shadow: out=(8, 768, 224, 224)
  cows: out=(8, 768, 224, 224)
  dance-twirl: out=(8, 768, 224, 224)
  dog: out=(8, 768, 224, 224)
  dogs-jump: out=(8, 768, 224, 224)
  drift-chicane: out=(8, 768, 224, 224)
  drift-straight: out=(8, 768, 224, 224)
  goat: out=(8, 768, 224, 224)
  gold-fish: out=(8, 768, 224, 224)
  horsejump-high: out=(8, 768, 224, 224)
  india: out=(8, 768, 224, 224)
  judo: out=(8, 768, 224, 224)
  kite-surf: out=(8, 768, 224, 224)
  lab-coat: out=(8, 768, 224, 224)
  libby: out=(8, 768, 224, 224)
  loading: out=(8, 768, 224, 224)
  mbike-trick: out=(8, 768, 224, 224)
  motocross-jump: out=(8, 768, 224, 224)
  paragliding-launch: out=(8, 768, 224, 224)
  parkour: out=(8, 768, 224, 224)
  pigs: out=(8, 768, 224, 224)
  scooter-black: out=(8, 768,

## Cell 6 — Adjacent Frame Cosine Similarity → Results Table & Save

In [9]:
# ── CELL 6 ───────────────────────────────────────────────────────────────────
# Adjacent frame cosine similarity → results table + save

def adjacent_cosine_sim(feats):
    """
    feats: (T, C, H, W)
    Returns mean cosine similarity across all adjacent pairs and spatial positions.
    """
    feats_norm = F.normalize(feats, dim=1)                      # (T, C, H, W)
    sim        = (feats_norm[:-1] * feats_norm[1:]).sum(dim=1)  # (T-1, H, W)
    return sim.mean().item()

rows = []
for clip_name in clips:
    path_2d = os.path.join(CACHE_DIR, f"{clip_name}_2d.pt")
    path_3d = os.path.join(CACHE_DIR, f"{clip_name}_3d.pt")
    if not os.path.exists(path_2d) or not os.path.exists(path_3d):
        print(f"  skip {clip_name} — inference cache missing")
        continue
    f2 = torch.load(path_2d, map_location="cpu")
    f3 = torch.load(path_3d, map_location="cpu")
    sim_2d = adjacent_cosine_sim(f2)
    sim_3d = adjacent_cosine_sim(f3)
    rows.append((clip_name, sim_2d, sim_3d))
    del f2, f3

# ── print table ───────────────────────────────────────────────────────────────
col_w   = max(len(n) for n, *_ in rows) + 2
header  = f"{'Clip':<{col_w}} {'2D AnyUp':>12} {'AnyUp3D':>12} {'Δ':>10}"
divider = "─" * len(header)
print(header)
print(divider)

sims_2d, sims_3d = [], []
for clip_name, s2, s3 in rows:
    print(f"{clip_name:<{col_w}} {s2:>12.4f} {s3:>12.4f} {s3-s2:>+10.4f}")
    sims_2d.append(s2)
    sims_3d.append(s3)

print(divider)
m2, std2 = np.mean(sims_2d), np.std(sims_2d)
m3, std3 = np.mean(sims_3d), np.std(sims_3d)
print(f"{'Mean ± Std':<{col_w}} {m2:.4f}±{std2:.4f} {m3:.4f}±{std3:.4f} {m3-m2:>+10.4f}")

# ── save ──────────────────────────────────────────────────────────────────────
result_path = os.path.join(OUTPUT_DIR, "temporal_cosine_sim.txt")
with open(result_path, "w") as f:
    f.write(header + "\n" + divider + "\n")
    for clip_name, s2, s3 in rows:
        f.write(f"{clip_name:<{col_w}} {s2:>12.4f} {s3:>12.4f} {s3-s2:>+10.4f}\n")
    f.write(f"\n{'Mean ± Std':<{col_w}} {m2:.4f}±{std2:.4f} {m3:.4f}±{std3:.4f} {m3-m2:>+10.4f}\n")

print(f"\nSaved → {result_path}")
# save locally for download
with open(os.path.join(LOCAL_OUT, "temporal_cosine_sim.txt"), "w") as f:
    f.write(header + "\n" + divider + "\n")
    for clip_name, s2, s3 in rows:
        f.write(f"{clip_name:<{col_w}} {s2:>12.4f} {s3:>12.4f} {s3-s2:>+10.4f}\n")
    f.write(f"\n{'Mean ± Std':<{col_w}} {m2:.4f}±{std2:.4f} {m3:.4f}±{std3:.4f} {m3-m2:>+10.4f}\n")
print(f"Local copy → {LOCAL_OUT}/temporal_cosine_sim.txt")

Clip                     2D AnyUp      AnyUp3D          Δ
─────────────────────────────────────────────────────────
bike-packing               0.8793       0.9744    +0.0952
blackswan                  0.9174       0.9712    +0.0538
bmx-trees                  0.8097       0.9279    +0.1182
breakdance                 0.8752       0.9782    +0.1030
camel                      0.9015       0.9725    +0.0710
car-roundabout             0.8934       0.9504    +0.0570
car-shadow                 0.9272       0.9666    +0.0394
cows                       0.9142       0.9778    +0.0636
dance-twirl                0.8689       0.9520    +0.0831
dog                        0.8066       0.9249    +0.1183
dogs-jump                  0.9164       0.9751    +0.0588
drift-chicane              0.9060       0.9745    +0.0685
drift-straight             0.8124       0.9012    +0.0888
goat                       0.8586       0.9520    +0.0934
gold-fish                  0.9110       0.9756    +0.0645
horsejump-high

NameError: name 'LOCAL_OUT' is not defined

## Cell 7 — PCA Visualization: 2D AnyUp row vs AnyUp3D row across T frames

In [ ]:
# ── CELL 7 ───────────────────────────────────────────────────────────────────
# PCA visualization: 2D AnyUp row vs AnyUp3D row across T frames

VIZ_CLIP   = list(clips.keys())[0]  # ← swap to any clip name from the table above

PCA_SAMPLE = 50_000   # max tokens used to FIT PCA; ↓ reduce if slow or OOM
                      # does not affect which frames are visualized, only fitting speed

f2 = torch.load(os.path.join(CACHE_DIR, f"{VIZ_CLIP}_2d.pt"), map_location="cpu")  # (8, 768, 224, 224)
f3 = torch.load(os.path.join(CACHE_DIR, f"{VIZ_CLIP}_3d.pt"), map_location="cpu")  # (8, 768, 224, 224)
T  = f2.shape[0]

def feats_to_pca_rgb(feats_2d, feats_3d):
    T, C, H, W = feats_2d.shape
    all_flat = np.concatenate([
        feats_2d.permute(0, 2, 3, 1).reshape(-1, C).numpy(),
        feats_3d.permute(0, 2, 3, 1).reshape(-1, C).numpy(),
    ], axis=0)
    idx = np.random.choice(len(all_flat), min(PCA_SAMPLE, len(all_flat)), replace=False)
    pca = PCA(n_components=3)
    pca.fit(all_flat[idx])

    def apply_pca(feats):
        flat = feats.permute(0, 2, 3, 1).reshape(-1, C).numpy()
        proj = pca.transform(flat)
        proj = (proj - proj.min()) / (proj.max() - proj.min() + 1e-8)
        return proj.reshape(T, H, W, 3)

    return apply_pca(feats_2d), apply_pca(feats_3d)

pca_2d, pca_3d = feats_to_pca_rgb(f2, f3)
del f2, f3

row_labels = ["2D AnyUp (frame-by-frame)", "AnyUp3D (full volume)"]
n_rows     = 2
label_col  = 0.08   # ← increase if labels are still cut off
fig, axes  = plt.subplots(n_rows, T, figsize=(3 * T + 1.5, 6))

for row_idx, (pca_imgs, label) in enumerate(zip([pca_2d, pca_3d], row_labels)):
    # row label using fig.text so axis("off") doesn't hide it
    fig.text(
        0.01,                                      # x position (left margin)
        1 - (row_idx + 0.5) / n_rows,             # y position centered on row
        label,
        va="center", ha="left",
        fontsize=10, fontweight="bold",
        rotation=90,
    )
    for t in range(T):
        ax = axes[row_idx, t]
        ax.imshow(pca_imgs[t])
        ax.axis("off")
        if row_idx == 0:
            ax.set_title(f"t={t}", fontsize=9)

fig.suptitle(f"PCA Feature Visualization — {VIZ_CLIP}", fontsize=12, y=1.02)
plt.tight_layout(rect=[0.06, 0, 1, 1])   # leave space on left for row labels; adjust 0.06 if needed

# ── save — THIS FILE is what you copy into your thesis ───────────────────────
viz_path = os.path.join(OUTPUT_DIR, f"pca_{VIZ_CLIP}.png")
plt.savefig(viz_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Thesis figure saved → {viz_path}")
print(f"Thesis table saved  → {os.path.join(OUTPUT_DIR, 'temporal_cosine_sim.txt')}")

In [ ]:
import shutil
shutil.make_archive("/content/exp2_thesis_outputs", "zip", LOCAL_OUT)
print("Download ready: /content/exp2_thesis_outputs.zip")